# EXP-R training: three seeds of Qwen3.5-4B (Design 7.3; DESIGN_ADDENDUM G and L; D220 to D223)

This notebook only calls the gate-E code. `scripts/ft_finetune.py train` trains each seed; `scripts/exp_r_memcheck.py`
(read-only) runs the D221 memory check once. Nothing is written inside the repo folder. A session writes only to Drive:
the run folders `models/ft-s42`, `ft-s43`, `ft-s44` (written by the train command) and `models/_exp_r/` (this
notebook's records: the pinned commit, the D221 verdict, one log per seed).

**Rules.** An NVIDIA **L4** only, never another GPU type (D220). Open the notebook from the pinned commit (Cell 1
prints the link). One session per seed at a time. `--grad-checkpointing` comes from the D221 verdict: never add or
remove it by hand. Never delete or rename a run folder; Cell 8 is the only restart (D222).

**Every session:** Cell 1, Cell 2, Cell 3 (every line PASS), then the cell that Cell 3 names.<br>
**Once, before seed 42:** Cell 4, then Cell 3 again, then Cell 5 in the same runtime.<br>
**After a disconnect:** a new L4 runtime; Cells 1, 2, 3; then the same train cell. It resumes from its last save
(every 50 weight updates and every validation).<br>
**Between seeds (D223):** Cell 2 prints the checks; Cell 3 lets the next seed start only when they pass. No answer is
generated or read between seeds, and no setting changes.

In [ ]:
# Cell 1: setup (G5, D222). Run it first on every new runtime. It ends with ALL PASS or FAIL lines.
L4_HOURS_CAP = {'off': 20.0, 'on': 26.0}   # hard stop over all EXP-R training sessions (D245)
AUTO_DISCONNECT = True                     # after a train command ends: flush Drive, then release the L4

import os, sys
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'   # D222: no __pycache__ in the repo folder, for every command
sys.dont_write_bytecode = True
import collections, glob, hashlib, importlib, json, math, platform, re, shutil, signal, subprocess, time, uuid
from datetime import datetime, timezone
from importlib import metadata, util
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/banglaindoorvqa-main'
ROOT = '/content/drive/MyDrive/BanglaIndoorVQA'
XR = f'{ROOT}/models/_exp_r'            # EXP-R records, outside every run folder (D246)
LOGS = f'{XR}/logs'
GATE_E = '5db2bfff3fd3e31b83f8c4865d37388f25f931f0'
QWEN_REV = '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'
EXP_R_FILES = ('notebooks/exp_r.ipynb', 'scripts/exp_r_memcheck.py')
SEEDS = ('ft-s42', 'ft-s43', 'ft-s44')
os.makedirs(LOGS, exist_ok=True)
SETUP_FAILS = []

def utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def check(ok, what, hint=''):
    print(('PASS  ' if ok else 'FAIL  ') + what + ('' if ok or not hint else '   -> ' + hint))
    if not ok:
        SETUP_FAILS.append(what)
    return ok

def read_json(path, default=None):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    except (OSError, ValueError):
        return default

def ver(*names):
    for n in names:
        try:
            return metadata.version(n)
        except metadata.PackageNotFoundError:
            pass
    return ''

def gpu_info():
    """(name, MiB in all, MiB in use) from nvidia-smi. The kernel never starts CUDA itself."""
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=60)
        name, total, used = [x.strip() for x in out.stdout.strip().splitlines()[0].split(',')]
        return name, int(float(total)), int(float(used))
    except Exception:
        return 'no GPU', 0, 0

def env_now():
    return {'gpu': gpu_info()[0], 'python': platform.python_version(), 'torch': ver('torch'),
            'transformers': ver('transformers'), 'peft': ver('peft'), 'fla': ver('flash-linear-attention', 'fla'),
            'causal_conv1d': ver('causal-conv1d', 'causal_conv1d'), 'torchao': ver('torchao')}

TOKEN = userdata.get('GITHUB_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

def git(*args, check_rc=True):
    r = subprocess.run(['git', '-C', REPO, *args], capture_output=True, text=True)
    if not check_rc:
        return r
    if r.returncode:
        raise SystemExit('git ' + args[0] + ' failed: ' + (r.stderr or r.stdout).replace(TOKEN or '***', '***')[-400:])
    return r.stdout.strip()

def is_ancestor(a, b):
    return git('merge-base', '--is-ancestor', a, b, check_rc=False).returncode == 0

# 1. The repo at the pinned EXP-R commit, never a newer one (D222)
if not os.path.isdir(REPO):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@github.com/blkeyd/banglaindoorvqa-main.git', REPO],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit('git clone failed: ' + (r.stderr or r.stdout).replace(TOKEN or '***', '***')[-400:])
git('fetch', '-q', '--tags', '--force', 'origin')
adds = {p: sorted(set(git('log', '--all', '--diff-filter=A', '--format=%H', '--', p).split())) for p in EXP_R_FILES}
if any(len(v) != 1 for v in adds.values()):
    raise SystemExit(f'Each EXP-R file must be added by exactly one commit; found {adds}. Ask in the EXP-R chat.')
a_, b_ = adds[EXP_R_FILES[0]][0], adds[EXP_R_FILES[1]][0]
PINNED = a_ if (a_ == b_ or is_ancestor(b_, a_)) else (b_ if is_ancestor(a_, b_) else None)
if PINNED is None:
    raise SystemExit('The two EXP-R files were added on different branches. Ask in the EXP-R chat.')
rec_path = f'{XR}/pinned_commit.txt'
rec = (open(rec_path).read().split() or [''])[0] if os.path.exists(rec_path) else ''
if not rec:
    with open(rec_path, 'w') as f:
        f.write(f'{PINNED}\n# the EXP-R commit (D222, D246), recorded {utc()} by the first EXP-R session\n')
    print(f'NOTE  first EXP-R session: pinned commit {PINNED} recorded in Drive models/_exp_r/pinned_commit.txt')
    rec = PINNED
if rec != PINNED:
    raise SystemExit(f'The EXP-R files now point to commit {PINNED[:12]}, but Drive records {rec[:12]} (D222). '
                     'Ask in the EXP-R chat; do not train.')
later = git('log', '--all', '--format=%h', f'^{PINNED}', '--', *EXP_R_FILES).split()
check(not later, 'no commit after the pinned one changes the EXP-R files' + (f' (found {later})' if later else ''),
      'the pin stays (D222): ask in the EXP-R chat')
git('checkout', '-q', '--detach', PINNED)
check(git('rev-parse', 'HEAD') == PINNED,
      f"repo at the pinned commit {PINNED[:12]}, detached (git describe: {git('describe', '--tags', check_rc=False).stdout.strip()})")
print(f'      open this notebook only from https://colab.research.google.com/github/blkeyd/banglaindoorvqa-main/blob/{PINNED}/notebooks/exp_r.ipynb')

# 2. Libraries (G5, D252): flash-linear-attention, transformers 5.18.0, peft 0.21.0; then torchao removed (D181).
#    flash-linear-attention keeps the version the D221 memory check ran with (D246).
mc = read_json(f'{XR}/memcheck.json') or {}
fla_ver, fla_from = (mc.get('env') or {}).get('fla') or '', 'the memory check (D246)'
if not fla_ver:
    v = str(((read_json(f'{ROOT}/models/ft-dry/ft_manifest.json') or {}).get('env') or {}).get('fla') or '')
    fla_ver, fla_from = (v, 'the Gate G dry run') if re.fullmatch(r'\d+(\.\d+)+', v) else ('', '')
print('installing: flash-linear-attention' + (f'=={fla_ver}' if fla_ver else '') + ', transformers 5.18.0, peft 0.21.0 ...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'flash-linear-attention=={fla_ver}' if fla_ver else 'flash-linear-attention',
                    'transformers==5.18.0', 'peft==0.21.0'], capture_output=True, text=True)
if r.returncode:
    raise SystemExit('pip install failed:\n' + (r.stderr or r.stdout)[-1500:])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], capture_output=True, text=True)
importlib.invalidate_caches()
ENV = env_now()
check(util.find_spec('torchao') is None and not ENV['torchao'], 'torchao is not installed (D181)')
check(ENV['peft'] == '0.21.0', f"peft is 0.21.0 (found {ENV['peft'] or 'none'}) (D171)")
check(ENV['transformers'] == '5.18.0', f"transformers is 5.18.0 (found {ENV['transformers'] or 'none'}) (D153)")
check(util.find_spec('fla') is not None and bool(ENV['fla']),
      f"flash-linear-attention {ENV['fla']} is installed (D152); "
      + (f'the version of {fla_from}' if fla_ver else 'no earlier version recorded, so the newest'))
check(util.find_spec('causal_conv1d') is None, 'causal_conv1d is not installed: its PyTorch fallback is the fixed state (D152)')
check(os.environ.get('PYTHONDONTWRITEBYTECODE') == '1' and sys.dont_write_bytecode, 'PYTHONDONTWRITEBYTECODE=1 here and for every command (D222)')
check(bool(os.environ.get('HF_TOKEN')) and bool(TOKEN), 'HF_TOKEN and GITHUB_TOKEN come from Colab Secrets')
check('L4' in ENV['gpu'], f"GPU: {ENV['gpu']} (D220: an NVIDIA L4, never another type)", 'Runtime, Change runtime type, L4 GPU')

# 3. This runtime, for the L4-hour count (Cell 2), and the runner that logs every command to Drive (D222)
if 'RUNTIME_ID' not in globals():
    RUNTIME_ID = uuid.uuid4().hex[:12]
    with open(f'{LOGS}/runtimes.log', 'a', encoding='utf-8') as f:
        f.write(f"[{utc()}] RUNTIME id={RUNTIME_ID} gpu={ENV['gpu'].replace(' ', '_')} commit={PINNED[:12]}\n")

def run_logged(cmd, log_path, header):
    """Run a command in the foreground. Show its output here and append every line, with its UTC time,
    to a Drive log file outside the run folder (D222). Returns (exit code, last lines, interrupted)."""
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONDONTWRITEBYTECODE='1', HF_HUB_DISABLE_PROGRESS_BARS='1')
    tail, interrupted, t0, synced = collections.deque(maxlen=400), False, time.time(), time.time()
    log_ok = [True]
    print('log: Drive ' + log_path.replace(ROOT + '/', ''))
    with open(log_path, 'a', encoding='utf-8') as log:
        def put(text, sync=False):
            if not log_ok[0]:
                return
            try:                               # a Drive hiccup never stops the command: the output stays on screen
                log.write(f'[{utc()}] {text}\n')
                if sync:
                    log.flush()
                    os.fsync(log.fileno())
            except OSError as exc:
                log_ok[0] = False
                print(f'WARNING: the Drive log stopped working ({exc}); the command goes on, its output is on screen only.',
                      flush=True)
        put(f'##### EXP-R SESSION {header}')
        put('$ ' + ' '.join(str(c) for c in cmd), sync=True)
        proc = subprocess.Popen([str(c) for c in cmd], cwd='/content', env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for ln in proc.stdout:
                ln = ln.rstrip('\n')
                print(ln, flush=True)
                tail.append(ln)
                put(ln, sync=time.time() - synced > 60)
                if time.time() - synced > 60:
                    synced = time.time()
            rc = proc.wait()
        except KeyboardInterrupt:
            interrupted = True
            proc.send_signal(signal.SIGINT)
            try:
                rc = proc.wait(timeout=180)
            except subprocess.TimeoutExpired:
                proc.kill()
                rc = proc.wait()
            put('interrupted by the user')
        put(f'##### EXIT {rc} after {(time.time() - t0) / 3600:.2f} h', sync=True)
    return rc, list(tail), interrupted

print()
print('ALL PASS: run Cell 2, then Cell 3.' if not SETUP_FAILS else f'{len(SETUP_FAILS)} FAIL: fix them first.')

In [ ]:
# Cell 2: where the three seeds stand, and the D223 checks between seeds. Read-only.
# It reads Drive only: no GPU, no repo, no torch. To use it on its own (a CPU runtime is enough):
#   from google.colab import drive; drive.mount('/content/drive')   then paste and run this cell.
# Printouts: counts, losses, times, memory and hashes only.
import glob, hashlib, json, math, os, re, shutil
from datetime import datetime, timezone
ROOT = globals().get('ROOT') or '/content/drive/MyDrive/BanglaIndoorVQA'
XR = f'{ROOT}/models/_exp_r'
SEEDS = ('ft-s42', 'ft-s43', 'ft-s44')
MANIFEST_FILE, ADAPTER_FILE = 'ft_manifest.json', 'adapter_model.safetensors'   # DESIGN_ADDENDUM G4
_LINE = re.compile(r'^\[(\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ)\] (.*)$')

def _rj(path, default=None):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    except (OSError, ValueError):
        return default

def _jsonl(path):
    recs, bad = [], 0
    if os.path.exists(path):
        with open(path, encoding='utf-8', errors='replace') as f:
            for ln in f:
                ln = ln.strip()
                if ln:
                    try:
                        recs.append(json.loads(ln))
                    except ValueError:
                        bad += 1
    return recs, bad

def _latest(recs, kind, key):
    """One record per key; a later line replaces an earlier one (a resumed session repeats steps)."""
    out = {}
    for r in recs:
        if r.get('kind') == kind:
            out[r[key]] = r
    return [out[k] for k in sorted(out)]

def _finite(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)

def _f(x, n=4):
    return f'{x:.{n}f}' if _finite(x) else str(x)

def _ts(s):
    return datetime.strptime(s, '%Y-%m-%dT%H:%M:%SZ').replace(tzinfo=timezone.utc)

def _sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

def log_sessions(path):
    """The sessions of one Drive log file: header fields, time of the first and last line, exit code."""
    out, cur = [], None
    if os.path.exists(path):
        with open(path, encoding='utf-8', errors='replace') as f:
            for ln in f:
                m = _LINE.match(ln.rstrip('\n'))
                if not m:
                    continue
                t, text = _ts(m.group(1)), m.group(2)
                if text.startswith('##### EXP-R SESSION'):
                    cur = dict(re.findall(r'(\w+)=(\S+)', text), first=t, last=t, exit=None)
                    out.append(cur)
                elif cur is not None:
                    cur['last'] = t
                    if text.startswith('##### EXIT'):
                        cur['exit'] = text.split()[2]
    return out

def l4_hours():
    """L4 hours of all EXP-R training runtimes: runtime start (Cell 1) to its last logged line; at least 0.1 each."""
    start, last = {}, {}
    rt = f'{XR}/logs/runtimes.log'
    if os.path.exists(rt):
        with open(rt, encoding='utf-8', errors='replace') as f:
            for ln in f:
                m = _LINE.match(ln.rstrip('\n'))
                if m and m.group(2).startswith('RUNTIME '):
                    kv = dict(re.findall(r'(\w+)=(\S+)', m.group(2)))
                    if 'L4' in kv.get('gpu', ''):
                        start.setdefault(kv.get('id'), _ts(m.group(1)))
                        last.setdefault(kv.get('id'), _ts(m.group(1)))
    for path in glob.glob(f'{XR}/logs/*.log'):
        if path.endswith('runtimes.log'):
            continue
        for s in log_sessions(path):
            rid = s.get('runtime')
            if rid not in start:
                start[rid], last[rid] = s['first'], s['last']
            else:
                last[rid] = max(last[rid], s['last'])
    return sum(max(0.1, (last[r] - start[r]).total_seconds() / 3600) for r in start), len(start)

def _view(man):
    """What must not change between seeds (D223): everything of the run except the seed."""
    fs = dict(man.get('ft_settings') or {})
    fs.pop('seed', None)
    g = lambda k: man.get(k) or {}
    return {'ft_settings without the seed': fs,
            'base model, commit, fixed settings': [g('base').get(k) for k in ('model_id', 'model_revision', 'settings_sha256')],
            'prompt': [g('prompt').get(k) for k in ('prompt_version', 'prompts_sha256', 'template_sha256')],
            'data': [g('data').get(k) for k in ('dataset_sha256', 'splits_sha256', 'train_ids_sha256', 'val_ids_sha256',
                                                'train_ids_sha256_run', 'val_ids_sha256_run')],
            'adapted layers': [g('lora').get(k) for k in ('targets_sha256', 'trainable_params')],
            'environment': {k: g('env').get(k) for k in ('gpu', 'torch', 'transformers', 'peft', 'fla', 'causal_conv1d',
                                                         'python', 'model_revision')},
            'plan': [g('plan').get(k) for k in ('total_updates', 'eval_points')],
            'code commit': str(g('git').get('commit') or '')[:12]}

def seed_report(run, mc, pinned, ref_man, ref_skipped):
    rd = f'{ROOT}/models/{run}'
    r = {'run': run, 'exists': os.path.isdir(rd), 'complete': False, 'd223': None, 'checks': [], 'info': {}}
    if not r['exists']:
        return r
    man = _rj(f'{rd}/{MANIFEST_FILE}', {}) or {}
    r['manifest'] = man
    res, plan = man.get('result') or {}, man.get('plan') or {}
    recs, bad = _jsonl(f'{rd}/train_log.jsonl')
    batches, updates = _latest(recs, 'batch', 'batch'), _latest(recs, 'update', 'update')
    erecs, ebad = _jsonl(f'{rd}/eval_log.jsonl')
    elog = _latest(erecs, 'eval', 'update')
    sess = log_sessions(f'{XR}/logs/{run}.log')
    r['complete'] = bool(man.get('complete'))
    evs = (res.get('evals') if r['complete'] else None) or [
        {'update': e['update'], 'val_loss': e.get('val_loss'), 'seconds': e.get('seconds')} for e in elog]
    recent = [u for u in updates[-200:] if _finite(u.get('wall_s'))]
    walls = [u['wall_s'] / max(1, u.get('examples') or 1) for u in recent]
    vsecs = [e.get('seconds') for e in evs if _finite(e.get('seconds'))]
    info = r['info']
    info.update(updates_logged=max((u['update'] for u in updates), default=0), batches_logged=len(batches),
                bad_lines=bad + ebad, sessions=len(sess), total_updates=plan.get('total_updates'),
                hours=sum((s['last'] - s['first']).total_seconds() for s in sess) / 3600,
                s_per_example=sum(walls) / len(walls) if walls else None,
                s_per_update=sum(u['wall_s'] for u in recent) / len(recent) if recent else None,
                s_per_validation=sum(vsecs) / len(vsecs) if vsecs else None,
                evals=[(e['update'], e['val_loss']) for e in evs],
                flags=sorted({s.get('grad_checkpointing', '?') for s in sess}),
                skipped=len(res.get('skipped') or {}))
    if not r['complete']:
        return r
    checks = r['checks']
    # 1. adapter and ft_manifest written, state/ removed
    ad = man.get('adapter') or {}
    w = f"{rd}/{ad.get('dir', 'adapter')}/{ad.get('file', ADAPTER_FILE)}"
    same = os.path.exists(w) and os.path.getsize(w) == ad.get('bytes') and _sha(w) == ad.get('sha256')
    left = [p for p in ('state',) if os.path.exists(f'{rd}/{p}')] + [os.path.basename(p) for p in glob.glob(f'{rd}/best_update*')]
    checks.append(('adapter and ft_manifest written, state/ removed', same and not left,
                   f"adapter sha256 {str(ad.get('sha256'))[:12]}, {(ad.get('bytes') or 0) / 2**20:.1f} MiB, "
                   + ('file matches' if same else 'FILE MISSING OR DIFFERENT') + (f"; left over: {', '.join(left)}" if left else '')))
    # 2. no NaN or infinite loss
    vals = ([b.get('loss') for b in batches] + [u.get('loss') for u in updates if u.get('batches')]
            + [u.get('grad_norm') for u in updates] + [e.get('val_loss') for e in evs])
    nonfinite = sum(1 for v in vals if not _finite(v))
    checks.append(('no NaN or infinite loss', nonfinite == 0 and bool(batches) and bool(evs),
                   f'{len(batches)} batch losses, {len(updates)} update losses and gradient norms, '
                   f'{len(evs)} validation losses; {nonfinite} not finite'))
    # 3. every planned validation present, or an early stop logged
    points, got = list(plan.get('eval_points') or []), [e['update'] for e in evs]
    stop, done = res.get('stopped'), res.get('updates_done')
    if stop:
        ok = bool(got) and got == [p for p in points if p <= stop.get('update', 0)] and got[-1] == stop.get('update') == done
        det = f"early stop logged after update {stop.get('update')} ({stop.get('reason')}); validations at {got}"
    else:
        ok = got == points and done == plan.get('total_updates')
        det = f'{len(got)} of {len(points)} planned validations, at {got}; {done} of {plan.get("total_updates")} updates'
    checks.append(('every planned validation present, or an early stop logged', ok, det))
    # 4. the same number of examples left out as the other seeds (and as the memory check predicts)
    refs = dict(ref_skipped)
    if isinstance(((mc or {}).get('predicted') or {}).get('train_over_cap'), int):
        refs = {'memory check predicts': mc['predicted']['train_over_cap'], **refs}
    n_skip = info['skipped']
    checks.append(('the same number of examples left out as the other seeds', bool(refs) and all(v == n_skip for v in refs.values()),
                   f'{n_skip} left out; ' + ', '.join(f'{k} {v}' for k, v in refs.items())))
    # 5. last validation loss below the first
    ok = len(evs) >= 2 and _finite(evs[0].get('val_loss')) and _finite(evs[-1].get('val_loss')) and evs[-1]['val_loss'] < evs[0]['val_loss']
    checks.append(('last validation loss below the first', ok,
                   f"{_f(evs[0].get('val_loss'))} after update {evs[0]['update']}, {_f(evs[-1].get('val_loss'))} after update {evs[-1]['update']}" if evs else 'no validation'))
    # 6. no setting changed between seeds (D223): everything but the seed, the pinned commit, the D221 flag
    diffs = [] if ref_man is None else [k for k, v in _view(ref_man).items() if _view(man)[k] != v]
    gi = man.get('git') or {}
    commit_ok = bool(pinned) and str(gi.get('commit') or '')[:12] == pinned[:12] and gi.get('dirty') is False
    flag_ok = bool(mc) and info['flags'] == [mc.get('verdict')]
    checks.append(('no setting changed: as ft-s42 except the seed; the pinned commit; the D221 flag in every session',
                   not diffs and commit_ok and flag_ok,
                   (f"differs from ft-s42 in: {', '.join(diffs)}; " if diffs else 'settings as ft-s42; ')
                   + f"commit {str(gi.get('commit') or '?')[:12]} ({'pinned' if commit_ok else 'NOT the pinned commit or not clean'}); "
                   + f"checkpointing in its sessions {info['flags']}, verdict {(mc or {}).get('verdict')}"))
    r['d223'] = all(c[1] for c in checks)
    # information, not D223: every example once per epoch, missing log lines
    per_epoch = {}
    for b in batches:
        per_epoch.setdefault(b.get('epoch'), []).extend(b.get('qa_ids') or [])
    info['epochs'] = {e: (len(ids), len(set(ids))) for e, ids in sorted(per_epoch.items(), key=lambda x: str(x[0]))}
    info['log_gaps'] = len(set(range(1, (done or 0) + 1)) - {u['update'] for u in updates})
    return r

def exp_r_status(quiet=False):
    mc = _rj(f'{XR}/memcheck.json')
    pinned = ((open(f'{XR}/pinned_commit.txt').read().split() or [''])[0]
              if os.path.exists(f'{XR}/pinned_commit.txt') else '')
    seeds, ref_man, skipped = {}, None, {}
    for run in SEEDS:
        man = _rj(f'{ROOT}/models/{run}/{MANIFEST_FILE}', {}) or {}
        if man.get('complete'):
            skipped[run] = len((man.get('result') or {}).get('skipped') or {})
    for run in SEEDS:
        others = {f'{k}': v for k, v in skipped.items() if k != run}
        seeds[run] = seed_report(run, mc, pinned, ref_man if run != 'ft-s42' else None, others)
        if run == 'ft-s42' and seeds[run]['exists']:
            ref_man = seeds[run].get('manifest')
    hours, n_rt = l4_hours()
    out = {'seeds': seeds, 'mc': mc, 'pinned': pinned, 'hours': hours, 'runtimes': n_rt,
           'failed': sorted(os.path.basename(p) for p in glob.glob(f'{XR}/failed/*'))}
    if quiet:
        return out
    print('=' * 78)
    print(f"EXP-R status   {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}   pinned commit {pinned[:12] or 'not recorded yet'}"
          f"   Drive free {shutil.disk_usage(ROOT).free / 2**30:.1f} GiB")
    if mc:
        pr = mc.get('predicted') or {}
        print(f"D221 memory check ({mc.get('written_utc')}): checkpointing {str(mc.get('verdict')).upper()}; peak reserved "
              f"with it off {(mc.get('off') or {}).get('peak_reserved_gib')} GiB (limit {mc.get('limit_gib')}); examples over the "
              f"cap: {pr.get('train_over_cap')} training, {pr.get('val_over_cap')} validation")
    else:
        print('D221 memory check: not run yet')
    print(f'L4 hours of EXP-R training so far: about {hours:.1f} in {n_rt} runtime(s)'
          + (f"; failed attempts moved aside (D222): {', '.join(out['failed'])}" if out['failed'] else ''))
    print('=' * 78)
    for run in SEEDS:
        s = seeds[run]
        i = s['info']
        if not s['exists']:
            print(f'{run}  not started')
            continue
        evs = ' | '.join(f'{u} {_f(v)}' for u, v in i['evals']) or 'none yet'
        spe = f"{i['s_per_example']:.2f} s per example" if i.get('s_per_example') else 'speed not known yet'
        if not s['complete']:
            print(f"{run}  IN PROGRESS   {i['updates_logged']} of {i['total_updates']} updates in the log; {spe}; "
                  f"{i['hours']:.2f} L4 hours in {i['sessions']} session(s); checkpointing {i['flags']}")
            print(f'        validation (update, loss): {evs}')
            continue
        man = s['manifest']
        res, ad = man.get('result') or {}, man.get('adapter') or {}
        stop = res.get('stopped')
        print(f"{run}  COMPLETE   {res.get('updates_done')} of {i['total_updates']} updates"
              + (f" (early stop after update {stop.get('update')})" if stop else '')
              + f"; best after update {ad.get('from_update')}, validation loss {_f(ad.get('val_loss'))}; "
              f"{i['hours']:.2f} L4 hours in {i['sessions']} session(s); {spe}")
        print(f'        validation (update, loss): {evs}')
        for name, ok, det in s['checks']:
            print(f"        D223  {'PASS' if ok else 'FAIL'}  {name}: {det}")
        ep = '; '.join(f'epoch {e}: {n} examples ({k} different)' for e, (n, k) in i.get('epochs', {}).items())
        print(f"        information: {ep}; weight updates missing from the log: {i.get('log_gaps')}; unreadable log lines: {i['bad_lines']}")
        nxt = SEEDS[SEEDS.index(run) + 1] if run != SEEDS[-1] else None
        print(f"        D223 for {run}: " + ('PASS' + (f'. {nxt} may start.' if nxt else '. All three seeds are trained.')
                                            if s['d223'] else 'FAIL. The next seed does not start: send this output to the EXP-R chat.'))
    return out

STATUS = exp_r_status()

In [ ]:
# Cell 3: preflight (D222). Read-only: it writes nothing anywhere. Run it after Cells 1 and 2 at the start of
# every session, and again before every train cell. Every line must be PASS.
ENV_DECISION = ''   # only when a decision row accepts an environment other than the Gate G dry run's, e.g. 'D247'

sys.path.insert(0, REPO + '/src')
FAILS = []

def line(ok, what, hint=''):
    print(('PASS  ' if ok else 'FAIL  ') + what + ('' if ok or not hint else '   -> ' + hint))
    if not ok:
        FAILS.append(what)
    return ok

def note(text):
    print('NOTE  ' + text)

def attempt(what, fn, hint=''):
    """fn returns (ok, detail). An error inside it is a FAIL, never a crash."""
    try:
        ok, detail = fn()
    except Exception as exc:
        ok, detail = False, f'{type(exc).__name__}: {str(exc)[:300]}'
    return line(ok, what + (f' ({detail})' if detail else ''), hint)

def sha12(rel):
    with open(f'{REPO}/{rel}', 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]

print('--- A. Repo (D222) ---')
line(git('rev-parse', 'HEAD') == PINNED, f'HEAD is the pinned commit {PINNED[:12]}', 'run Cell 1')
line(is_ancestor(GATE_E, 'HEAD') and git('rev-parse', 'gate-E^{commit}') == GATE_E,
     'the tag gate-E (5db2bfff3fd3) is in the history of HEAD')
dirty = git('status', '--porcelain', '--untracked-files=all', '--ignored')
line(not dirty, 'clean tree; nothing written inside the repo folder'
     + (f" (found: {', '.join(dirty.splitlines()[:3])})" if dirty else ''), 'Runtime, Restart session, then Cells 1 to 3')
TRAIN_FILES = ('src/finetune.py', 'scripts/ft_finetune.py', 'src/harness.py', 'src/prompts.py', 'src/images.py',
               'src/normalise.py', 'src/cost_ledger.py', 'configs/finetune.yaml', 'configs/models.yaml',
               'configs/conditions.yaml', 'configs/prompts.yaml', 'probes/FT-VAL400.ids.txt')
chg = git('diff', '--name-only', GATE_E, 'HEAD', '--', *TRAIN_FILES).split()
line(not chg and all(os.path.exists(f'{REPO}/{p}') for p in TRAIN_FILES),
     f'the {len(TRAIN_FILES)} files training uses are byte-identical to gate-E' + (f' (changed: {chg})' if chg else ''),
     'the training code is frozen at gate-E: ask in the EXP-R chat')
KNOWN = {'src/finetune.py': '8c4719aeae57', 'scripts/ft_finetune.py': 'c2c5f9d84e04', 'src/harness.py': '0a11abd8a4ec',
         'configs/finetune.yaml': 'df91bcf93a6e', 'configs/models.yaml': '36f8da1d8120', 'configs/conditions.yaml': '40897bfcf50c'}
got = {p: sha12(p) for p in KNOWN}
line(got == KNOWN, 'the files this notebook was written for: ' + ', '.join(f"{p.split('/')[-1]} {v}" for p, v in got.items()),
     'send this line to the EXP-R chat')
DRY_FILES = ('src/finetune.py', 'scripts/ft_finetune.py', 'src/harness.py', 'src/prompts.py', 'src/images.py',
             'configs/prompts.yaml')
chg = git('diff', '--name-only', '0e36f2fc2264', 'HEAD', '--', *DRY_FILES).split()
line(not chg, 'the six files the Gate G dry run used are unchanged since its commit 0e36f2fc2264'
     + (f' (changed: {chg})' if chg else ''), 'a new dry run is needed (G5): ask in the EXP-R chat')

def changed_lines_only(path, pattern):
    body = [ln for ln in git('diff', '-U0', '0e36f2fc2264', 'HEAD', '--', path).splitlines()
            if ln[:1] in '+-' and not ln.startswith(('+++', '---'))]
    return all(re.match(pattern, ln) for ln in body), len(body)

ok1, n1 = changed_lines_only('configs/finetune.yaml', r'^[+-]\s*(peft_version:|#|$)')
ok2, n2 = changed_lines_only('configs/models.yaml', r'^[+-]\s*(revision:|#|$)')
line(ok1 and ok2, f'since the dry run, finetune.yaml changed only peft_version ({n1} lines) and models.yaml only '
     f'the revision lines ({n2} lines) (D171, D155)', 'ask in the EXP-R chat')
note('files changed since gate-E: ' + (', '.join(git('diff', '--name-only', GATE_E, 'HEAD').split()) or 'none'))

print('--- B. Gate G, settings and the model (D176) ---')
import harness as H, finetune as F, images as imglib
line(H.FT_MANIFEST == MANIFEST_FILE and H.ADAPTER_FILE == ADAPTER_FILE,
     f'run-folder file names as Cell 2 reads them ({H.FT_MANIFEST}, {H.ADAPTER_FILE})')
ft = models = m = D = None

def _settings():
    global ft, models, m
    ft = F.load_ft(Path(REPO) / 'configs' / 'finetune.yaml')
    models = H.load_models(Path(REPO) / 'configs' / 'models.yaml')
    m = models[ft['base_model']]
    ok = ft['base_model'] == 'qwen' and m.get('revision') == QWEN_REV and str(ft.get('peft_version')) == '0.21.0'
    return ok, f"FT scripts {F.FT_VERSION}; {m.get('model_id')} at {str(m.get('revision'))[:12]}; peft_version {ft.get('peft_version')}"

attempt('configs hold the Design 7.3 values, Qwen pinned at 851bf6e806ef and peft 0.21.0', _settings)
dry = read_json(f'{ROOT}/models/ft-dry/{MANIFEST_FILE}') or {}
dry_rev = str((dry.get('base') or {}).get('model_revision'))
line(dry_rev == QWEN_REV, f'the Gate G dry run trained on the same Qwen commit ({dry_rev[:12]})')
rep = read_json(f'{ROOT}/models/ft-dry/gate_g_report.json') or {}
vd = rep.get('verdict') or {}
line(rep.get('gate_g_passed') is True and len(vd) == 8 and all(v is True for v in vd.values())
     and rep.get('script_version') == F.FT_VERSION and rep.get('peft') == '0.21.0' and bool(rep.get('seconds')),
     f"Gate G report {rep.get('utc')}: passed, {sum(v is True for v in vd.values())} of {len(vd)} checks true, script "
     f"{rep.get('script_version')} (current {F.FT_VERSION}), peft {rep.get('peft')}")

def _hub():
    from huggingface_hub import HfApi
    sha = HfApi().model_info('Qwen/Qwen3.5-4B', revision=QWEN_REV, token=os.environ.get('HF_TOKEN')).sha
    return sha == QWEN_REV, f'commit {sha[:12]}'

attempt('the token reads Qwen/Qwen3.5-4B at the pinned commit on Hugging Face', _hub)
GIT = H.git_info(Path(REPO))
line(GIT.get('gate_e') is True and GIT.get('dirty') is False,
     f"what the train command reads: gate_e {GIT.get('gate_e')}, dirty {GIT.get('dirty')}")

print('--- C. Data (D172, D173, D175) ---')

def _data():
    global D
    D = F.define_data(Path(ROOT) / 'data' / 'banglaindoorvqa.json', Path(ROOT) / 'data' / 'splits.csv',
                      val_n=int(ft['val_n']), dry_n=int(ft['runs']['ft-dry']['n_train']))
    F.check_data_manifest(D, Path(ROOT) / 'data' / 'ft' / F.DATA_MANIFEST)
    return (len(D['train']), len(D['val'])) == (13668, 400), \
        f"{len(D['train'])} training QA, {len(D['val'])} validation items; hashes equal data/ft/{F.DATA_MANIFEST}"

attempt('the training and validation data are those the data command fixed (G2)', _data)

def _seed_requirements():
    F.check_seed_requirements(ft, m, GIT, Path(ROOT), Path(REPO), D, ver('peft'))
    return True, ''

attempt("the train command's own seed check (D176: gate-E, clean tree, commit, peft, Gate G report, FT-VAL400)",
        _seed_requirements)

def _images():
    man = imglib.read_manifest(Path(ROOT) / 'data' / 'images_1024')
    need = F.images_of(D['train'] + D['val'])
    miss = sum(1 for f in need if f not in man)
    return miss == 0, f'{len(need) - miss} of {len(need)} prepared'

attempt('every training and validation image is prepared (D175)', _images)

print('--- D. Environment (D220, D222, D223) ---')
gpu_name, gpu_total, gpu_used = gpu_info()
line('L4' in gpu_name, f'GPU: {gpu_name}, {gpu_total} MiB (D220)', 'Runtime, Change runtime type, L4 GPU; never another type')
now = env_now()
line(now == ENV, 'libraries unchanged since Cell 1')
line(not now['torchao'] and now['peft'] == '0.21.0' and now['transformers'] == '5.18.0' and bool(now['fla'])
     and not now['causal_conv1d'], f"torchao absent, peft {now['peft']}, transformers {now['transformers']}, "
     f"flash-linear-attention {now['fla']}, causal_conv1d absent (D152, D153, D171, D181)")
MC = read_json(f'{XR}/memcheck.json')
EKEYS = ('gpu', 'python', 'torch', 'transformers', 'peft', 'fla', 'causal_conv1d')
if MC:
    ref = MC.get('env') or {}
    diff = [f'{k} {ref.get(k)} -> {now.get(k)}' for k in EKEYS if ref.get(k) != now.get(k)]
    line(not diff, 'the environment equals that of the D221 memory check'
         + (f" (differs: {'; '.join(diff)})" if diff else ''), 'D223: no setting changes; ask in the EXP-R chat')
else:
    denv = dry.get('env') or {}
    diff = [f'{k} {denv.get(k)} -> {now.get(k)}' for k in ('gpu', 'torch', 'transformers', 'peft')
            if str(denv.get(k)) != str(now.get(k))]
    if diff and ENV_DECISION:
        note(f"the environment differs from the Gate G dry run ({'; '.join(diff)}); accepted by {ENV_DECISION}")
    else:
        line(not diff, 'the environment equals that of the Gate G dry run (GPU, torch, transformers, peft)'
             + (f" (differs: {'; '.join(diff)})" if diff else ''),
             'ask in the EXP-R chat; a decision row then goes into ENV_DECISION at the top of this cell')
    if str(denv.get('python')) != now['python']:
        note(f"Python {now['python']}; the dry run had {denv.get('python')} (information)")
free_local = shutil.disk_usage('/content').free / 2**30
line(free_local >= 25, f'local disk: {free_local:.0f} GiB free (25 needed for the model files and the image copies)')
free_drive = shutil.disk_usage(ROOT).free / 2**30
line(free_drive >= 3, f'Drive: {free_drive:.1f} GiB free (about 1.5 GiB needed while a seed trains)')
if free_drive < 20:
    note('less than 20 GiB free in Drive. The resume state (about 0.37 GiB) is rewritten 43 times per seed; '
         'if Drive keeps the replaced copies in its bin, empty the bin between sessions.')

print('--- E. Runs (D221, D222, D223) ---')
ST = exp_r_status(quiet=True)
seeds = ST['seeds']
NEXT = next((r for r in SEEDS if not seeds[r]['complete']), None)
k_next = SEEDS.index(NEXT) if NEXT else len(SEEDS)
late = [r for r in SEEDS[k_next + 1:] if seeds[r]['exists']]
line(not late, f"seed order 42, 43, 44: next is {NEXT or 'none (all three complete)'}"
     + (f'; started out of order: {late}' if late else ''), 'ask in the EXP-R chat')
for r in SEEDS[:k_next]:
    line(seeds[r]['d223'] is True, f'D223 checks of {r}: ' + ('PASS' if seeds[r]['d223'] else 'FAIL (see Cell 2)'),
         'the next seed does not start: send the Cell 2 output to the EXP-R chat')
answers = sorted({p for pat in ('outputs/main/qwen_ft_s4*', 'outputs/pilot/qwen_ft_s4*', 'outputs/smoke/*/qwen_ft_s4*')
                  for p in glob.glob(f'{ROOT}/{pat}')})
line(not answers or NEXT is None, 'no answer of a seed adapter exists while seeds are still training (D223)'
     + (f' (found {len(answers)} folder(s))' if answers else ''), 'ask in the EXP-R chat')
if MC:
    line(MC.get('verdict') in ('off', 'on') and MC.get('pinned_commit') == PINNED,
         f"D221 verdict: gradient checkpointing {str(MC.get('verdict')).upper()} for all three seeds "
         f"({MC.get('written_utc')}, commit {str(MC.get('pinned_commit'))[:12]})")
elif NEXT == 'ft-s42' and not seeds['ft-s42']['exists']:
    note('no D221 verdict yet: Cell 4 (memory check) runs before ft-s42')
else:
    line(False, 'a seed folder exists but there is no D221 verdict', 'ask in the EXP-R chat')

STATE_PROBLEM = {}

def check_state(run):
    """Read-only: does the newest saved state load, and does it fit the run (D222)?"""
    rd = f'{ROOT}/models/{run}'
    man = read_json(f'{rd}/{MANIFEST_FILE}') or {}
    sp = f'{rd}/state/last.pt'
    logged = seeds[run]['info'].get('updates_logged', 0)
    if not os.path.exists(sp):
        if logged:
            return False, (f'the saved state is missing although train_log.jsonl records weight updates up to {logged}'
                           + (' (state/last.pt.tmp exists)' if os.path.exists(sp + '.tmp') else ''))
        return True, ('no saved state yet: the train command starts this seed at update 0'
                      + (' (the session before ended before its first save)' if man else ''))
    try:
        import torch
        d = torch.load(sp, map_location='cpu', weights_only=True)
        prog, lora, lm = json.loads(d['progress']), d['lora'], man.get('lora') or {}
        n_w = sum(int(t.numel()) for t in lora.values())
        finite = all(bool(torch.isfinite(t).all()) for t in lora.values())
        n_opt = len((d.get('opt') or {}).get('state') or {})
        best, best_ok = prog.get('best'), True
        if best:
            w = f"{rd}/{best['dir']}/{ADAPTER_FILE}"
            best_ok = os.path.exists(w) and _sha(w) == best.get('sha256')
        upd, n_t = int(d['update']), len(lora)
        bad = [name for name, wrong in (('the update number', upd != int(prog.get('update', -1))),
                                        ('the number of LoRA tensors', len(lora) != lm.get('n_tensors')),
                                        ('the number of LoRA weights', n_w != lm.get('trainable_params')),
                                        ('a LoRA weight that is not finite', not finite),
                                        ('the AdamW state', n_opt not in (0, len(lora))),
                                        ('the best checkpoint it names', not best_ok)) if wrong]
        del d, lora
        if bad:
            return False, 'the saved state does not fit the run: ' + ', '.join(bad)
        return True, (f"the saved state loads: after update {upd} of {(man.get('plan') or {}).get('total_updates')}, "
                      f"{n_t} LoRA tensors with {n_w:,} weights, all finite; "
                      + (f"best checkpoint (update {best['update']}) present" if best else 'no best checkpoint yet')
                      + '; the train command resumes from here')
    except Exception as exc:
        return False, f'the saved state cannot be loaded ({type(exc).__name__}: {str(exc)[:200]})'

if NEXT and seeds[NEXT]['exists']:
    ok, detail = check_state(NEXT)
    if not ok:
        STATE_PROBLEM[NEXT] = detail
    line(ok, f'{NEXT}: {detail}', 'D222: run Cell 3 once more; if this line stays FAIL, Cell 8 starts this seed again from scratch')
    sess = log_sessions(f'{LOGS}/{NEXT}.log')
    if sess and sess[-1].get('runtime') != RUNTIME_ID and sess[-1].get('exit') is None:
        age = (datetime.now(timezone.utc) - sess[-1]['last']).total_seconds() / 60
        line(age >= 12, f"{NEXT}: no other session trains it (its log was last written {age:.0f} min ago by runtime "
             f"{sess[-1].get('runtime')}, without an exit line)",
             'one session per run folder: end the other session (Runtime, Manage sessions), wait until 12 minutes '
             'have passed, then run Cell 3 again')

# L4 hours: used so far, and the projection to the end of the third seed, against the hard stop (Design 12.2)
secs = rep.get('seconds') or {}
t_tr, t_val = float(secs.get('train_per_example') or 'nan'), float(secs.get('val_per_item') or 'nan')
on_ = (MC or {}).get('verdict') == 'on'
ratio = (((MC or {}).get('projection') or {}).get('time_ratio_on_off') if on_ else None) or 1.0
n_tr = len(D['train']) if D else 13668
pts = F.make_plan(n_tr, ft, {'epochs': int(ft['epochs'])})['eval_points'] if ft else []
seed_h = (n_tr * int(ft['epochs'] if ft else 2) * t_tr * ratio + len(pts) * int(ft['val_n'] if ft else 400) * t_val) / 3600
done_h = [seeds[r]['info']['hours'] for r in SEEDS if seeds[r]['complete'] and seeds[r]['info']['hours'] > 1]
seed_h = max(done_h) if done_h else seed_h
left_h = 0.0 if MC else 0.2
for r in SEEDS:
    s, i = seeds[r], seeds[r]['info']
    if s['complete']:
        continue
    if s['exists'] and i.get('updates_logged'):
        spu = i.get('s_per_update') or 16 * t_tr * ratio
        spv = i.get('s_per_validation') or int(ft['val_n'] if ft else 400) * t_val
        left_h += (max(0, (i['total_updates'] or 0) - i['updates_logged']) * spu
                   + sum(1 for p in pts if p > i['updates_logged']) * spv) / 3600 + 0.25
    else:
        left_h += seed_h + 0.25
cap = L4_HOURS_CAP['on' if on_ else 'off']
proj = ST['hours'] + left_h
line(math.isfinite(proj) and proj <= cap,
     f"L4 hours: used about {ST['hours']:.1f} in {ST['runtimes']} runtime(s); projected total about {proj:.1f} of the "
     f"hard stop {cap:.1f} ({'with' if on_ else 'without'} checkpointing)",
     'Design 12.2: ask in the EXP-R chat before going over; do not start this session')
apps = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader,nounits'],
                      capture_output=True, text=True).stdout.strip()
line(not apps and gpu_used < 256, f'no process holds GPU memory ({gpu_used} MiB in use)'
     + (f'; processes: {apps.splitlines()[:2]}' if apps else ''),
     'Runtime, Restart session, then Cells 1 to 3 (this notebook never starts CUDA in its own kernel)')

PREFLIGHT = {'ok': not FAILS and bool(MC) and NEXT is not None,
             'ok_for_memcheck': not FAILS and not MC and NEXT == 'ft-s42' and not seeds['ft-s42']['exists'],
             'next': NEXT, 'runtime': RUNTIME_ID, 't': time.time(), 'state_problem': STATE_PROBLEM}
print()
if FAILS:
    print(f'{len(FAILS)} FAIL: fix them before Cell 4 or a train cell.')
elif NEXT is None:
    print('ALL PASS. All three seeds are complete: no train cell to run. Send the Cell 2 output to the EXP-R chat.')
elif not MC:
    print('ALL PASS. Next: Cell 4 (the D221 memory check, once), then Cell 3 again, then Cell 5 (ft-s42).')
else:
    print(f'ALL PASS. Next: Cell {5 + SEEDS.index(NEXT)}, train {NEXT}'
          + (' (it resumes from the saved state).' if seeds[NEXT]['exists'] else ' (a new start).'))


def train_seed(run):
    """One train command in the foreground, output also in Drive models/_exp_r/logs/<run>.log (D222)."""
    pf = globals().get('PREFLIGHT') or {}
    if pf.get('runtime') != RUNTIME_ID or not pf.get('ok'):
        raise SystemExit('Run Cell 3 in this runtime first; it must end with ALL PASS.')
    if pf.get('next') != run:
        raise SystemExit(f"Cell 3 names {pf.get('next')} as the next seed, not {run} (D223: 42, then 43, then 44).")
    if time.time() - pf['t'] > 3600:
        raise SystemExit('Cell 3 ran more than an hour ago: run it again.')
    mc_ = read_json(f'{XR}/memcheck.json') or {}
    if mc_.get('verdict') not in ('off', 'on'):
        raise SystemExit('No D221 verdict: run Cell 4 first.')
    gc_on = mc_['verdict'] == 'on'
    cmd = [sys.executable, '-u', f'{REPO}/scripts/ft_finetune.py', 'train', '--root', ROOT, '--repo', REPO,
           '--run', run] + (['--grad-checkpointing'] if gc_on else [])
    header = (f"run={run} runtime={RUNTIME_ID} commit={PINNED[:12]} grad_checkpointing={'on' if gc_on else 'off'} "
              + ' '.join(f"{k}={str(v).replace(' ', '_') or 'none'}" for k, v in ENV.items()))
    PREFLIGHT['ok'] = False                    # one preflight per train command
    rc, tail, interrupted = run_logged(cmd, f'{LOGS}/{run}.log', header)
    text = '\n'.join(tail)
    print()
    if rc == 0 and 'NOT COMPLETE' not in text:
        exp_r_status()
    elif interrupted:
        print('Interrupted. The train command keeps everything up to its last save (every 50 weight updates and at '
              'each validation). To go on: Cells 1 (new runtime only), 2 and 3, then this cell again (D222).')
    elif 'out of memory' in text.lower():
        print('STOP (D221, D224): the seed ran out of GPU memory. Do not add or remove --grad-checkpointing, although '
              'the message above suggests it: D221 fixes the flag for all three seeds. Send the whole output to the '
              'EXP-R chat. The saved state is kept.')
    else:
        print(f'STOP: the train command ended with exit code {rc}. Send the full error text, the traceback, the last '
              '20 lines of output and "Colab L4" to the EXP-R chat. Delete nothing: the saved state is kept.')
    if AUTO_DISCONNECT and not interrupted:
        print('Releasing the L4 in 2 minutes. To keep the runtime, interrupt this cell now (the stop button).')
        try:
            time.sleep(120)
        except KeyboardInterrupt:
            print('Runtime kept. End the session with Cell 9 when you are done.')
            return rc
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()
    return rc

## Once, before seed 42: the D221 memory check
It decides `--grad-checkpointing` for all three seeds. Off if the peak reserved memory with it off is at most 20.0 GiB
on the 8 longest training examples; otherwise on, if loss and gradients match the off run (each loss within 2^-8
relative, the summed LoRA gradient within 1% relative L2). Anything else stops training (D224).

In [ ]:
# Cell 4: the D221 memory check. Once, before seed 42, on the L4, after Cell 3 ends with ALL PASS.
# About 10 minutes: the model loads, the training and validation images are copied to the local disk (the
# train command reuses these copies in this runtime), then 25 training passes on the 8 longest examples.
MC_PATH = f'{XR}/memcheck.json'
if os.path.exists(MC_PATH):
    mc_ = read_json(MC_PATH) or {}
    print(f"The D221 verdict exists ({mc_.get('written_utc')}): gradient checkpointing "
          f"{str(mc_.get('verdict')).upper()} for all three seeds. Nothing to do.")
else:
    pf = globals().get('PREFLIGHT') or {}
    if pf.get('runtime') != RUNTIME_ID or not pf.get('ok_for_memcheck'):
        raise SystemExit('Run Cell 3 in this runtime first; it must end with ALL PASS and name Cell 4.')
    cmd = [sys.executable, '-u', f'{REPO}/scripts/exp_r_memcheck.py', '--root', ROOT, '--repo', REPO,
           '--image-cache', '/content/ft_images_1024']
    header = f'run=memcheck runtime={RUNTIME_ID} commit={PINNED[:12]} ' + ' '.join(
        f"{k}={str(v).replace(' ', '_') or 'none'}" for k, v in ENV.items())
    rc, tail, interrupted = run_logged(cmd, f'{LOGS}/memcheck.log', header)
    res = next((json.loads(t.split(' ', 1)[1]) for t in reversed(tail) if t.startswith('MEMCHECK_RESULT ')), None)
    PREFLIGHT['ok_for_memcheck'] = False
    if rc == 0 and res and res.get('verdict') in ('off', 'on'):
        rec = dict(res, pinned_commit=PINNED, runtime=RUNTIME_ID, env=ENV, env_decision=ENV_DECISION,
                   written_utc=utc())
        with open(MC_PATH + '.tmp', 'w', encoding='utf-8') as f:
            json.dump(rec, f, indent=1)
        os.replace(MC_PATH + '.tmp', MC_PATH)
        print(f"\nRecorded in Drive models/_exp_r/memcheck.json: gradient checkpointing {res['verdict'].upper()} "
              'for all three seeds (D221). Next: Cell 3 again, then Cell 5 (ft-s42) in this runtime.')
    else:
        print(f'\nSTOP (D221, D224): no verdict was recorded (exit code {rc}). No seed is trained. '
              'Send the whole output above to the EXP-R chat.')

## Training, one cell per seed (D220 to D223)
Run only the cell that Cell 3 names. Keep the browser tab open while a seed trains. When the train command ends, the
cell prints the D223 checks and releases the L4 after 2 minutes (AUTO_DISCONNECT in Cell 1).

In [ ]:
# Cell 5: train seed 42 (D220 to D223), in the foreground. About 6 L4 hours without checkpointing, more with it.
# After a disconnect: a new L4 runtime, Cells 1, 2, 3, then this cell again. The train command resumes.
train_seed('ft-s42')

In [ ]:
# Cell 6: train seed 43. Cell 3 lets it start only when the D223 checks of ft-s42 pass (Cell 2 prints them).
# Before it starts: no answer is generated or read, and no setting is changed (D223).
train_seed('ft-s43')

In [ ]:
# Cell 7: train seed 44. Cell 3 lets it start only when the D223 checks of ft-s42 and ft-s43 pass.
train_seed('ft-s44')

## Only under D222: start a seed again from scratch
Only when Cell 3, run twice, says the saved state of the seed cannot be loaded or is missing. Ask in the EXP-R chat if
in doubt.

In [ ]:
# Cell 8: D222 restart of ONE seed from scratch. Only when Cell 3 printed, in two runs, that the saved state of
# this seed cannot be loaded or is missing. Nothing is deleted: the run folder moves to models/_exp_r/failed/.
# Design 7.3: a seed that fails a second time is reported with the seeds that ran, not restarted again.
RESTART_RUN = ''      # 'ft-s42', 'ft-s43' or 'ft-s44'
CONFIRM = ''          # type exactly: D222 restart <run>, for example 'D222 restart ft-s43'
if not RESTART_RUN:
    print('Nothing to do: RESTART_RUN is empty.')
else:
    pf = globals().get('PREFLIGHT') or {}
    problem = (pf.get('state_problem') or {}).get(RESTART_RUN)
    if pf.get('runtime') != RUNTIME_ID or not problem:
        raise SystemExit(f'Cell 3 in this runtime did not report a saved state of {RESTART_RUN} that cannot be '
                         'loaded. Nothing was moved.')
    if CONFIRM != f'D222 restart {RESTART_RUN}':
        raise SystemExit(f"Set CONFIRM = 'D222 restart {RESTART_RUN}' to move the folder. Nothing was moved.")
    earlier = sorted(os.path.basename(p) for p in glob.glob(f'{XR}/failed/{RESTART_RUN}_*'))
    if earlier:
        raise SystemExit(f'{RESTART_RUN} was restarted once already ({earlier}). Design 7.3: a seed that fails '
                         'twice is reported, not restarted. Ask in the EXP-R chat. Nothing was moved.')
    dst = f"{XR}/failed/{RESTART_RUN}_{utc().replace('-', '').replace(':', '')}"
    os.makedirs(f'{XR}/failed', exist_ok=True)
    os.rename(f'{ROOT}/models/{RESTART_RUN}', dst)
    with open(f'{LOGS}/restarts.log', 'a', encoding='utf-8') as f:
        f.write(f"[{utc()}] D222 RESTART {RESTART_RUN}: {problem}; folder moved to {dst.replace(ROOT + '/', '')} "
                f'by runtime {RUNTIME_ID}\n')
    PREFLIGHT['ok'] = False
    print(f"Moved models/{RESTART_RUN} to {dst.replace(ROOT + '/', '')}. Run Cell 3 again; the train cell of "
          f'{RESTART_RUN} then starts it at update 0 (D222).')

## End a session by hand

In [ ]:
# Cell 9: end a session by hand, when no train cell released the L4: flush Drive, then release the runtime.
from google.colab import runtime
drive.flush_and_unmount()
print('Drive flushed. Releasing the runtime.')
runtime.unassign()